# ADB joint-loss baseline

Compare joint losses across seven scenarios. The simplified ADB model covers the Kyrgyz Republic and Tajikistan over three years, with USD 70m for earthquakes and USD 10m for precipitation per country. Earthquake risk uses public calibration; precipitation risk and shared-event severity are model assumptions. [Model and sources](README.md#model-and-sources).

Run the cells from top to bottom. Keep the scenario configurations fixed when comparing methods.

| Scenario | Purpose |
|---|---|
| `adb-base` | Published structure and earthquake calibration |
| `adb-independent` | Independent countries; unchanged country risks |
| `adb-common-shock` | Double the shared earthquake rate; unchanged country risks |
| `adb-rain-stress` | Double precipitation event rates |
| `adb-rain-cover` | USD 60m earthquake / USD 20m precipitation per country |
| `kaz-kgz-base` | Hypothetical Kazakhstan-Kyrgyz Republic contract |
| `tjk-uzb-base` | Hypothetical Tajikistan-Uzbekistan contract |

The ADB cases measure term principal losses. The World Bank regional cases measure annual losses and payment rates.


In [ ]:
import csv
import json
from pathlib import Path
import subprocess
import sys

from IPython.display import Markdown, display
from qjlb.cli import write_csv

ROOT = Path.cwd()
if not (ROOT / "scenarios/adb-base.json").is_file():
    raise RuntimeError("Open this notebook from the repository folder.")

SCENARIOS = {path.stem: path for path in sorted((ROOT / "scenarios").glob("*.json"))}
OUTPUT = ROOT / "results/scenarios"
OUTPUT.mkdir(parents=True, exist_ok=True)
SAMPLES = 8192
REPEATS = 10
SEED = 1000


def command(*args):
    return subprocess.run(
        [sys.executable, "-m", "qjlb.cli", *map(str, args)],
        cwd=ROOT,
        capture_output=True,
        text=True,
    )


def show(rows):
    columns = list(rows[0])
    lines = [" | ".join(columns), " | ".join("---" for _ in columns)]
    lines += [" | ".join(str(row[key]) for key in columns) for row in rows]
    display(Markdown("\n".join(lines)))

## 1. Check the calibration

Compare annual earthquake risk with the public figures. Small expected-loss differences come from rounding and the 25% payout grid.


In [ ]:
from qjlb.bond import BondModel

model = BondModel.load(ROOT / "scenarios/adb-base.json")
terms = json.loads((ROOT / "data/adb_terms.json").read_text())
calibration_rows = []
for axis, country in ((1, "KGZ"), (0, "TJK")):
    marginal = model.earthquake_probabilities.sum(axis=axis)
    tail = terms["annual_earthquake_exceedance_probabilities"][country]
    published = [
        tail[0],
        terms["published_annual_earthquake_expected_loss_fraction"][country],
        tail[-1],
    ]
    calculated = [
        1 - marginal[0],
        sum(i * p / 4 for i, p in enumerate(marginal)),
        marginal[-1],
    ]
    for label, source, value in zip(
        ("Attachment", "Expected loss / EQ limit", "Exhaustion"), published, calculated
    ):
        calibration_rows.append(
            {
                "Country": country,
                "Annual measure": label,
                "Published (%)": f"{100 * source:.4f}",
                "Model (%)": f"{100 * value:.4f}",
            }
        )
show(calibration_rows)

## 2. Run the baselines

Run four sampling methods and the exact reference for each scenario. Adjust `SAMPLES` and `REPEATS` above.

The tables show exact scenario results and relative root-mean-square error (RMSE) for `adb-base` across repeats. Lower error is better. All targets and timings are saved in `baseline-summary.csv`; one-off times include preparation.

ADB calculations apply three-year principal caps. World Bank calculations use annual row rates.


In [ ]:
baseline_rows = []
bond_rows = []
regional_rows = []
for name, config in SCENARIOS.items():
    folder = OUTPUT / name / "baseline"
    result = command(
        "run",
        "--config",
        config,
        "--output",
        folder,
        "--samples",
        SAMPLES,
        "--repeats",
        REPEATS,
        "--seed",
        SEED,
    )
    if result.returncode:
        raise RuntimeError(result.stderr)
    with (folder / "summary.csv").open() as stream:
        baseline_rows.extend(
            {"scenario": name, **row} for row in csv.DictReader(stream)
        )
    reference = json.loads((folder / "reference.json").read_text())["estimates"]
    values = {key: row["estimate"] for key, row in reference.items()}
    if "term_expected_principal_loss_usd" in values:
        bond_rows.append(
            {
                "Scenario": name,
                "Term expected loss (USD)": f"{values['term_expected_principal_loss_usd']:,.0f}",
                "Both attach (%)": f"{100 * values['term_joint_attachment_probability']:.3f}",
                "Loss > USD 80m (%)": f"{100 * values['term_portfolio_loss_exceedance_probability']:.3f}",
                "Both exhaust (%)": f"{100 * values['term_both_bonds_exhausted_probability']:.5f}",
            }
        )
    else:
        regional_rows.append(
            {
                "Scenario": name,
                "Annual economic loss (USD)": f"{values['annual_economic_loss_usd']:,.0f}",
            }
        )

write_csv(OUTPUT / "baseline-summary.csv", baseline_rows)
display(Markdown("**ADB: three-year principal losses**"))
show(bond_rows)
display(Markdown("**World Bank: annual economic losses**"))
show(regional_rows)

labels = {
    "term_expected_principal_loss_usd": "Expected principal loss",
    "term_joint_attachment_probability": "Both bonds incur loss",
    "term_portfolio_loss_exceedance_probability": "Combined loss > USD 80m",
    "term_portfolio_stop_loss_usd": "Expected loss above USD 80m",
    "term_both_bonds_exhausted_probability": "Both bonds exhausted",
}
method_labels = {
    "mc": "Monte Carlo",
    "qmc": "Sobol",
    "stratified": "Stratified",
    "importance": "Importance",
}
base_errors = {
    (row["target"], row["method"]): float(row["relative_rmse"])
    for row in baseline_rows
    if row["scenario"] == "adb-base"
}
accuracy_rows = [
    {
        "Target": label,
        **{
            title: f"{100 * base_errors[target, method]:.2f}"
            for method, title in method_labels.items()
        },
    }
    for target, label in labels.items()
]
display(Markdown("**ADB base: relative RMSE (%)**"))
show(accuracy_rows)
print("Full results: results/scenarios/baseline-summary.csv")

## 3. Create example submissions

Create one Monte Carlo results file per scenario. Adapt [example.py](example.py) for your method. Rare outcomes can have large errors or no observed hits at the default sample budget.


In [ ]:
EXAMPLE_DIR = OUTPUT / "example-submissions"
for name, config in SCENARIOS.items():
    result = subprocess.run(
        [
            sys.executable,
            "example.py",
            "--config",
            str(config),
            "--output",
            str(EXAMPLE_DIR / f"{name}.json"),
        ],
        cwd=ROOT,
        capture_output=True,
        text=True,
    )
    if result.returncode:
        raise RuntimeError(result.stderr)
print(
    f"Created {len(SCENARIOS)} example results files in results/scenarios/example-submissions."
)

## 4. Validate your results

Set `SUBMISSION_DIR` to your JSON folder. Use scenario names as filenames, such as `adb-base.json`.

`RTOL = None` reports errors. `RTOL = 0.01` requires all five estimates to be within 1%; a zero reference requires a zero estimate. Missing or invalid files are listed.


In [ ]:
SUBMISSION_DIR = EXAMPLE_DIR  # Change this to your team's results folder.
RTOL = None


def validate_scenarios(submission_dir, rtol=None):
    rows = []
    for name, config in SCENARIOS.items():
        source = Path(submission_dir) / f"{name}.json"
        row = {
            "scenario": name,
            "status": "missing",
            "max_relative_error": None,
            "detail": "",
        }
        folder = OUTPUT / name / "validation"
        # Remove prior comparisons so an invalid or missing file cannot leave an old pass.
        for previous in (folder / "report.json", folder / "comparison.csv"):
            previous.unlink(missing_ok=True)
        if source.is_file():
            args = ["validate", source, "--config", config, "--output", folder]
            if rtol is not None:
                args.extend(["--rtol", rtol])
            result = command(*args)
            if result.returncode in (0, 1):
                report = json.loads((folder / "report.json").read_text())
                row["status"] = (
                    "compared"
                    if rtol is None
                    else ("pass" if report["within_tolerance"] else "fail")
                )
                errors = [item["relative_error"] for item in report["comparisons"]]
                row["max_relative_error"] = max(
                    (error for error in errors if error is not None), default=None
                )
            else:
                row.update(
                    status="invalid", detail=result.stderr.strip().splitlines()[-1]
                )
        rows.append(row)
    write_csv(OUTPUT / "validation-summary.csv", rows)
    return rows


validation_rows = validate_scenarios(SUBMISSION_DIR, RTOL)
show(
    [
        {
            "Scenario": row["scenario"],
            "Status": row["status"].capitalize(),
            "Largest relative error (%)": "n/a"
            if row["max_relative_error"] is None
            else f"{100 * row['max_relative_error']:.2f}",
            "Detail": row["detail"],
        }
        for row in validation_rows
    ]
)
print(
    "Errors and intervals: results/scenarios/<scenario>/validation/comparison.csv"
)
print("Validation summary: results/scenarios/validation-summary.csv")